In [4]:
pip -q install gradio groq pandas

In [5]:
import os
import sys
import gradio as gr
from groq import Groq

In [6]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

cliente = Groq(api_key=GROQ_API_KEY)

In [26]:
import pandas as pd
import os

nombre_archivo = "/content/componentes_electronicos.csv"

def buscar_en_inventario(termino_busqueda):
    if not os.path.exists(nombre_archivo):
        return "El archivo de inventario no está disponible."

    termino_busqueda = termino_busqueda.lower()

    try:
        df = pd.read_csv(nombre_archivo)
        if df.empty:
            return "El inventario está completamente vacío."

        total_productos = len(df)

        col_precio = "Precio Unitario ($)"
        col_stock = "Existencias Actuales"
        col_camino = "Próximas Existencias (En camino)"

        if col_precio in df.columns:
            df[col_precio] = df[col_precio].astype(str).str.replace(r'[^\d.]', '', regex=True)
            df[col_precio] = pd.to_numeric(df[col_precio], errors='coerce')

        if col_stock in df.columns:
            df[col_stock] = pd.to_numeric(df[col_stock], errors='coerce')

        if col_camino in df.columns:
            df[col_camino] = pd.to_numeric(df[col_camino], errors='coerce')

        if any(p in termino_busqueda for p in ["caro", "mas alto", "mayor precio", "expensive"]):
            if col_precio in df.columns and not df[col_precio].dropna().empty:
                idx_max = df[col_precio].idxmax()
                return f"--- ANÁLISIS DE INVENTARIO ---\nEl producto más caro es:\n{df.loc[idx_max].to_dict()}"

        if any(p in termino_busqueda for p in ["barato", "economico", "mas bajo", "cheap"]):
            if col_precio in df.columns and not df[col_precio].dropna().empty:
                idx_min = df[col_precio].idxmin()
                return f"--- ANÁLISIS DE INVENTARIO ---\nEl producto más barato es:\n{df.loc[idx_min].to_dict()}"

        if any(p in termino_busqueda for p in ["menos stock", "menos existencia", "agotarse", "critico"]):
            if col_stock in df.columns and not df[col_stock].dropna().empty:
                idx_min_stock = df[col_stock].idxmin()
                return f"--- ANÁLISIS DE INVENTARIO ---\nEl producto con menor existencia es:\n{df.loc[idx_min_stock].to_dict()}"

        if any(p in termino_busqueda for p in ["camino", "proximas", "pedido"]):
            if col_camino in df.columns:
                df_camino = df[df[col_camino] > 0]
                if not df_camino.empty:
                    return f"--- PRODUCTOS EN CAMINO ---\n" + df_camino.head(10).to_csv(index=False)

        palabras_clave = [p for p in termino_busqueda.split() if len(p) > 2]
        if palabras_clave:
            mascara = df.astype(str).apply(lambda x: x.str.lower().str.contains('|'.join(palabras_clave), na=False)).any(axis=1)
            df_filtrado = df[mascara]
            if not df_filtrado.empty:
                return f"--- COINCIDENCIAS ENCONTRADAS ---\n" + df_filtrado.head(15).to_csv(index=False)

        return (
            f"--- ESTADÍSTICAS GLOBALES DEL INVENTARIO ---\n"
            f"Cantidad total de productos distintos registrados: {total_productos}\n\n"
            f"Muestra de las primeras filas del inventario:\n{df.head(15).to_csv(index=False)}"
        )

    except Exception as e:
        return f"Error al analizar el inventario con Pandas: {str(e)}"

historial = []

print("¡Bienvenido a InventarioBot! Escribe tu pregunta sobre el inventario o 'terminar' para salir.\n")

while True:
    mensaje = input("Usuario: ")

    if mensaje.strip().lower() == 'terminar':
        print("InventarioBot: ¡Terminado!.")
        break

    if not mensaje.strip():
        continue

    datos_relevantes = buscar_en_inventario(mensaje)

    instrucciones_sistema = (
        "Eres InventarioBot, un chatbot experto en electrónica y gestión de inventarios. "
        "Tu objetivo es responder cualquier pregunta relacionada con el inventario de componentes electrónicos "
        "de forma clara, concisa y resumida. Evita introducciones largas e ve directo al grano. "
        "Usa negritas y viñetas cuando sea útil.\n\n"
        "A continuación tienes los datos extraídos del inventario mediante Pandas. "
        "Utilízalos rigurosamente para responder al usuario:\n"
        f"{datos_relevantes}"
    )

    mensajes_groq = [{
        "role": "system",
        "content": instrucciones_sistema
    }]

    for msg in historial[-6:]:
        mensajes_groq.append({"role": msg["role"], "content": msg["content"]})

    mensajes_groq.append({"role": "user", "content": mensaje})

    try:
        respuesta_stream = cliente.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=mensajes_groq,
            stream=True
        )

        historial.append({"role": "user", "content": mensaje})
        print("InventarioBot: ", end="", flush=True)
        texto_acumulado = ""

        for chunk in respuesta_stream:
            if chunk.choices and len(chunk.choices) > 0:
                delta_content = chunk.choices[0].delta.content
                if delta_content:
                    texto_acumulado += delta_content
                    print(delta_content, end="", flush=True)

        historial.append({"role": "assistant", "content": texto_acumulado})
        print("\n")

    except Exception as e:
        error_msg = f" Error al conectar con Groq. Detalles: {str(e)}"
        print(f"\nInventarioBot: {error_msg}\n")
        historial.append({"role": "user", "content": mensaje})
        historial.append({"role": "assistant", "content": error_msg})


¡Bienvenido a InventarioBot! Escribe tu pregunta sobre el inventario o 'terminar' para salir.

Usuario: Necesito saber si hay resistencias o algún capacitor disponible, ¿cuántos quedan?
InventarioBot: Aquí tienes la disponibilidad inmediata de **Resistencias** y **Capacitores**:

*   **Resistencias** (5 artículos):
    *   **Resistencias LED Rojo**: **231** unidades
    *   **Resistencias ESP32**: **386** unidades
    *   **Resistencias 1N4148**: **57** unidades
    *   **Resistencias 10k Ohm** (COMP-0079): **105** unidades
    *   **Resistencias 10k Ohm** (COMP-0084): **399** unidades

*   **Capacitores** (8 artículos):
    *   **Capacitores ATmega328P** (COMP-0017): **287** unidades
    *   **Capacitores NE555** (COMP-0020): **493** unidades
    *   **Capacitores 1N4148** (COMP-0042): **0** unidades (**Agotado**)
    *   **Capacitores 100nF** (COMP-0047): **199** unidades
    *   **Capacitores DHT11**: **341** unidades
    *   **Capacitores 2N2222** (COMP-0057): **214** unidades
    